# cow-csvw

Integrated CSV to RDF converter, using CSVW and nanopublications.

---
*Auto-generated from `codemeta.json`*

## 3. Upload your CSV dataset

Click the upload button and select your CSV file.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import io, os

upload = widgets.FileUpload(accept='.csv', multiple=False)
display(upload)
print('Upload your CSV file above, then run the next cell.')

## 4. Preview your data

In [ ]:
import pandas as pd
if upload.value:
    val = list(upload.value.values())[0] if isinstance(upload.value, dict) else upload.value[0]
    fname = val['metadata']['name'] if isinstance(upload.value, dict) else val['name']
    raw = bytes(val['content'])
    df = pd.read_csv(io.BytesIO(raw))
    df.to_csv(fname, index=False)
    print(f'Loaded: {fname} — {len(df)} rows, {len(df.columns)} columns')
    print(df.head())
else:
    print('No file uploaded yet.')

## 5. Convert to RDF

In [ ]:
import subprocess, sys, os

# Install cow-csvw if not already installed
subprocess.run([sys.executable, '-m', 'pip', 'install', 'cow-csvw', 'rdflib', '-q'])

# Convert CSV to RDF
result = subprocess.run(['python', '-m', 'cowsay', '--dataset', fname], capture_output=True, text=True)
if not result.stdout and not result.stderr:
    result = subprocess.run([sys.executable, '-m', 'cowsay', '--dataset', fname], capture_output=True, text=True)
print(result.stdout or result.stderr or 'Conversion attempted')

rdf_file = fname.replace('.csv', '.nq')
import rdflib
g = rdflib.Graph()
if os.path.exists(rdf_file):
    g.parse(rdf_file)
    print(f'Total triples: {len(g)}')
    for s,p,o in list(g)[:5]: print(s,p,o)
else:
    print('RDF file not found — cowsay may need different syntax')
    print('Files in directory:', os.listdir('.'))

## 6. Compare CSV vs RDF

In [ ]:
print('=== Original CSV ===')
print(f'Rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
print(df.head(3).to_string())
print('\n=== Generated RDF ===')
print(f'Total triples: {len(g)}')
print(f'Unique subjects: {len(set(s for s,p,o in g))}')
print(f'Unique predicates: {len(set(p for s,p,o in g))}')
for s,p,o in list(g)[:3]: print(f'  {str(s)[-30:]} | {str(p)[-30:]} | {str(o)[:30]}')
print('\nEach CSV row became multiple RDF triples.')